<a href="https://colab.research.google.com/github/Jubaida-78/Smart-Washing-Machine-Fuzzy-Logic/blob/main/Smart_Washing_Machine_Advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install scikit-fuzzy

#
# SMART WASHING MACHINE - FUZZY LOGIC SYSTEM (ADVANCED)
# =-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl
import matplotlib.pyplot as plt



print("="*60)
print("     SMART WASHING MACHINE-FUZZY LOGIC SYSTEM")
print("="*60)


print("\n ENTER CRISP INPUTS (follow the ranges strictly)\n")

print("Cloth Type:")
print("   1 = Delicate")
print("   2 = Normal")
print("   3 = Heavy")
while True:
    cloth_input = float(input("→ Enter Cloth Type [1, 2, or 3]: "))
    if 1 <= cloth_input <= 3:
        break
    print("  ERROR: Must be 1, 2, or 3 only!")

print("\nLoad Size: range 0 to 15 kg")
print("   Small  : 0 – 5 kg")
print("   Medium : 5 – 10 kg")
print("   Large  : 10 – 15 kg")
while True:
    load_input = float(input("→ Enter Load Size in kg [0 to 15]: "))
    if 0 <= load_input <= 15:
        break
    print("   ERROR: Must be between 0 and 15!")

print("\nDirtiness: range 0 to 10")
print("   Low    : 0 – 3")
print("   Medium : 3 – 7")
print("   High   : 7 – 10")
while True:
    dirt_input = float(input("→ Enter Dirtiness [0 to 10]: "))
    if 0 <= dirt_input <= 10:
        break
    print("    ERROR: Must be between 0 and 10!")

print("\nWater Hardness: range 0 to 500 ppm")
print("   Soft   : 0 – 166 ppm")
print("   Normal : 166 – 333 ppm")
print("   Hard   : 333 – 500 ppm")
while True:
    water_input = float(input("→ Enter Water Hardness in ppm [0 to 500]: "))
    if 0 <= water_input <= 500:
        break
    print("   ERROR: Must be between 0 and 500!")


#cloth_type = ctrl.Antecedent(np.arange(1, 3.01, 0.01), 'cloth_type')
#load_size = ctrl.Antecedent(np.arange(0, 15.01, 0.01), 'load_size')
#dirtiness = ctrl.Antecedent(np.arange(0, 10.01, 0.01), 'dirtiness')
#water_hardness = ctrl.Antecedent(np.arange(0, 500.01, 1), 'water_hardness')
#washing_intensity = ctrl.Consequent(np.arange(0, 100.01, 0.5), 'washing_intensity')
cloth_type = ctrl.Antecedent(np.linspace(1, 3, 201), 'cloth_type')
load_size = ctrl.Antecedent(np.linspace(0, 15, 1501), 'load_size')
dirtiness = ctrl.Antecedent(np.linspace(0, 10, 1001), 'dirtiness')
water_hardness = ctrl.Antecedent(np.linspace(0, 500, 501), 'water_hardness')
washing_intensity = ctrl.Consequent(np.linspace(0, 100, 1001), 'washing_intensity')


# STEP 3: MEMBERSHIP FUNCTIONS(trianguLlar)


cloth_type['delicate'] = fuzz.trimf(cloth_type.universe, [1.0, 1.0, 2.0])
cloth_type['normal']   = fuzz.trimf(cloth_type.universe, [1.0, 2.0, 3.0])
cloth_type['heavy']    = fuzz.trimf(cloth_type.universe, [2.0, 3.0, 3.0])

load_size['small']  = fuzz.trimf(load_size.universe, [0.0,  0.0,  7.5])
load_size['medium'] = fuzz.trimf(load_size.universe, [0.0,  7.5, 15.0])
load_size['large']  = fuzz.trimf(load_size.universe, [7.5, 15.0, 15.0])

dirtiness['low']    = fuzz.trimf(dirtiness.universe, [0.0, 0.0,  5.0])
dirtiness['medium'] = fuzz.trimf(dirtiness.universe, [0.0, 5.0, 10.0])
dirtiness['high']   = fuzz.trimf(dirtiness.universe, [5.0, 10.0, 10.0])

water_hardness['soft']   = fuzz.trimf(water_hardness.universe, [0.0,   0.0, 250.0])
water_hardness['normal'] = fuzz.trimf(water_hardness.universe, [0.0, 250.0, 500.0])
water_hardness['hard']   = fuzz.trimf(water_hardness.universe, [250.0, 500.0, 500.0])

washing_intensity['gentle'] = fuzz.trimf(washing_intensity.universe, [0.0,   0.0,  50.0])
washing_intensity['normal'] = fuzz.trimf(washing_intensity.universe, [0.0,  50.0, 100.0])
washing_intensity['strong'] = fuzz.trimf(washing_intensity.universe, [50.0, 100.0, 100.0])

#rules........
rule1 = ctrl.Rule(
    cloth_type['heavy'] & load_size['large'] & dirtiness['high'],
    washing_intensity['strong']
)
rule2 = ctrl.Rule(
    cloth_type['delicate'] & water_hardness['hard'],
    washing_intensity['gentle']
)
rule3 = ctrl.Rule(
    cloth_type['normal'] & load_size['medium'] & dirtiness['medium'],
    washing_intensity['normal']
)


rule4 = ctrl.Rule(
    cloth_type['normal'] & load_size['large'] & dirtiness['high'],
    washing_intensity['strong']
)
rule5 = ctrl.Rule(
    cloth_type['delicate'] & dirtiness['low'],
    washing_intensity['gentle']
)
rule6 = ctrl.Rule(
    cloth_type['heavy'] & dirtiness['medium'],
    washing_intensity['normal']
)
rule7 = ctrl.Rule(
    cloth_type['normal'] & water_hardness['hard'],
    washing_intensity['normal']
)


#building fuzzy system........
washing_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6,rule7])
washing_sim  = ctrl.ControlSystemSimulation(washing_ctrl)

washing_sim.input['cloth_type']     = cloth_input
washing_sim.input['load_size']      = load_input
washing_sim.input['dirtiness']      = dirt_input
washing_sim.input['water_hardness'] = water_input

washing_sim.compute()
crisp_output = washing_sim.output['washing_intensity']


if cloth_input <= 1.5:#label........
    cloth_label = "Delicate"
elif cloth_input <= 2.5:
    cloth_label = "Normal"
else:
    cloth_label = "Heavy"


if load_input <= 5:
    load_label = "Small"
elif load_input <= 10:
    load_label = "Medium"
else:
    load_label = "Large"


if dirt_input <= 3:
    dirt_label = "Low"
elif dirt_input <= 7:
    dirt_label = "Medium"
else:
    dirt_label = "High"


if water_input <= 166:
    water_label = "Soft"
elif water_input <= 333:
    water_label = "Normal"
else:
    water_label = "Hard"


if crisp_output < 34:
    out_label = "GENTLE"
elif crisp_output < 67:
    out_label = "NORMAL"
else:
    out_label = "STRONG"


print("\n")#printing result.....
print("=" * 60)
print("     SMART WASHING MACHINE - FUZZY LOGIC RESULT")
print("=" * 60)
print()
print(" CRISP INPUTS............................")
print(f"  Cloth Type     = {cloth_input}         → {cloth_label}")
print(f"  Load Size      = {load_input} kg      → {load_label}")
print(f"  Dirtiness      = {dirt_input}         → {dirt_label}")
print(f"  Water Hardness = {water_input} ppm   → {water_label}")
print()
print("  FUZZIFICATION ─────────────────────────────────────")
mu_del = fuzz.interp_membership(cloth_type.universe,
         fuzz.trimf(cloth_type.universe,[1,1,2]), cloth_input)
mu_nor = fuzz.interp_membership(cloth_type.universe,
         fuzz.trimf(cloth_type.universe,[1,2,3]), cloth_input)
mu_hvy = fuzz.interp_membership(cloth_type.universe,
         fuzz.trimf(cloth_type.universe,[2,3,3]), cloth_input)
print(f"  Cloth  → μ(Delicate)={mu_del:.2f}, μ(Normal)={mu_nor:.2f}, μ(Heavy)={mu_hvy:.2f}")

mu_sm = fuzz.interp_membership(load_size.universe,
        fuzz.trimf(load_size.universe,[0,0,7.5]), load_input)
mu_md = fuzz.interp_membership(load_size.universe,
        fuzz.trimf(load_size.universe,[0,7.5,15]), load_input)
mu_lg = fuzz.interp_membership(load_size.universe,
        fuzz.trimf(load_size.universe,[7.5,15,15]), load_input)
print(f"  Load   → μ(Small)={mu_sm:.2f}, μ(Medium)={mu_md:.2f}, μ(Large)={mu_lg:.2f}")

mu_lo = fuzz.interp_membership(dirtiness.universe,
        fuzz.trimf(dirtiness.universe,[0,0,5]), dirt_input)
mu_me = fuzz.interp_membership(dirtiness.universe,
        fuzz.trimf(dirtiness.universe,[0,5,10]), dirt_input)
mu_hi = fuzz.interp_membership(dirtiness.universe,
        fuzz.trimf(dirtiness.universe,[5,10,10]), dirt_input)
print(f"  Dirt   → μ(Low)={mu_lo:.2f}, μ(Medium)={mu_me:.2f}, μ(High)={mu_hi:.2f}")

mu_so = fuzz.interp_membership(water_hardness.universe,
        fuzz.trimf(water_hardness.universe,[0,0,250]), water_input)
mu_no = fuzz.interp_membership(water_hardness.universe,
        fuzz.trimf(water_hardness.universe,[0,250,500]), water_input)
mu_ha = fuzz.interp_membership(water_hardness.universe,
        fuzz.trimf(water_hardness.universe,[250,500,500]), water_input)
print(f"  Water  → μ(Soft)={mu_so:.2f}, μ(Normal)={mu_no:.2f}, μ(Hard)={mu_ha:.2f}")


print()
print("  ── RULE INFERENCE (Mamdani - MIN operator) ──────────")
r1 = min(mu_hvy, mu_lg, mu_hi)
r2 = min(mu_del, mu_ha)
r3 = min(mu_nor, mu_md, mu_me)
r4 = min(mu_nor, mu_lg, mu_hi)
r5 = min(mu_del, mu_lo)
r6 = min(mu_hvy, mu_me)
r7 = min(mu_nor, mu_ha)
print(f"  Rule1 (Heavy∧Large∧High  → Strong) : α = {r1:.2f}")
print(f"  Rule2 (Delicate∧Hard     → Gentle) : α = {r2:.2f}")
print(f"  Rule3 (Normal∧Medium∧Med → Normal) : α = {r3:.2f}")
print(f"  Rule4 (Normal∧Large∧High → Strong) : α = {r4:.2f}")
print(f"  Rule5 (Delicate∧Low      → Gentle) : α = {r5:.2f}")
print(f"  Rule6 (Heavy∧Medium      → Normal) : α = {r6:.2f}")
print(f"  Rule7 (Normal∧Hard       → Normal) : α = {r7:.2f}")
print()
print("  Defuzzification Method : Centroid (Centre of Gravity)")


print()
print("  CRISP OUTPUT....................................")
print(f"  Washing Intensity = {crisp_output:.4f} %")
print(f"  Washing Label     = {out_label}")
print()
print("=" * 60)
print(f"  FINAL ANSWER: Crisp Output = {crisp_output:.4f}% → {out_label}")
print("=" * 60)

#Graph
fig, axes = plt.subplots(nrows=5, figsize=(11, 22))

#Graph 1:Cloth Type
u_c = cloth_type.universe
axes[0].plot(u_c, fuzz.trimf(u_c,[1,1,2]), 'b-', lw=2.5, label='Delicate')
axes[0].plot(u_c, fuzz.trimf(u_c,[1,2,3]), 'g-', lw=2.5, label='Normal')
axes[0].plot(u_c, fuzz.trimf(u_c,[2,3,3]), 'r-', lw=2.5, label='Heavy')
axes[0].axvline(x=cloth_input, color='black', lw=2, linestyle='--',
                label=f'Input={cloth_input} ({cloth_label})')
axes[0].fill_between(u_c, fuzz.trimf(u_c,[1,1,2]), alpha=0.08, color='blue')
axes[0].fill_between(u_c, fuzz.trimf(u_c,[1,2,3]), alpha=0.08, color='green')
axes[0].fill_between(u_c, fuzz.trimf(u_c,[2,3,3]), alpha=0.08, color='red')
axes[0].set_title('Cloth Type Membership Function', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cloth Type  (1=Delicate, 2=Normal, 3=Heavy)')
axes[0].set_ylabel('Membership Degree  μ(x)')
axes[0].set_xlim([0.8, 3.2]); axes[0].set_ylim([-0.05, 1.15])
axes[0].legend(loc='upper right'); axes[0].grid(True, alpha=0.3)

#Graph 2:Load Type
u_l = load_size.universe
axes[1].plot(u_l, fuzz.trimf(u_l,[0,0,7.5]),  'b-', lw=2.5, label='Small (0-7.5kg)')
axes[1].plot(u_l, fuzz.trimf(u_l,[0,7.5,15]), 'g-', lw=2.5, label='Medium (0-15kg)')
axes[1].plot(u_l, fuzz.trimf(u_l,[7.5,15,15]),'r-', lw=2.5, label='Large (7.5-15kg)')
axes[1].axvline(x=load_input, color='black', lw=2, linestyle='--',
                label=f'Input={load_input}kg ({load_label})')
axes[1].fill_between(u_l, fuzz.trimf(u_l,[0,0,7.5]),   alpha=0.08, color='blue')
axes[1].fill_between(u_l, fuzz.trimf(u_l,[0,7.5,15]),  alpha=0.08, color='green')
axes[1].fill_between(u_l, fuzz.trimf(u_l,[7.5,15,15]), alpha=0.08, color='red')
axes[1].set_title('Load Size Membership Function', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Load Size (kg)  [Range: 0–15 kg]')
axes[1].set_ylabel('Membership Degree  μ(x)')
axes[1].set_xlim([-0.5, 15.5]); axes[1].set_ylim([-0.05, 1.15])
axes[1].legend(loc='upper right'); axes[1].grid(True, alpha=0.3)

#Graph 3:Dritiness
u_d = dirtiness.universe
axes[2].plot(u_d, fuzz.trimf(u_d,[0,0,5]),   'b-', lw=2.5, label='Low (0-5)')
axes[2].plot(u_d, fuzz.trimf(u_d,[0,5,10]),  'g-', lw=2.5, label='Medium (0-10)')
axes[2].plot(u_d, fuzz.trimf(u_d,[5,10,10]), 'r-', lw=2.5, label='High (5-10)')
axes[2].axvline(x=dirt_input, color='black', lw=2, linestyle='--',
                label=f'Input={dirt_input} ({dirt_label})')
axes[2].fill_between(u_d, fuzz.trimf(u_d,[0,0,5]),   alpha=0.08, color='blue')
axes[2].fill_between(u_d, fuzz.trimf(u_d,[0,5,10]),  alpha=0.08, color='green')
axes[2].fill_between(u_d, fuzz.trimf(u_d,[5,10,10]), alpha=0.08, color='red')
axes[2].set_title('Dirtiness Membership Function', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Dirtiness Level  [Range: 0–10]')
axes[2].set_ylabel('Membership Degree  μ(x)')
axes[2].set_xlim([-0.3, 10.3]); axes[2].set_ylim([-0.05, 1.15])
axes[2].legend(loc='upper right'); axes[2].grid(True, alpha=0.3)

#Graph 4:Water Hardness
u_w = water_hardness.universe
axes[3].plot(u_w, fuzz.trimf(u_w,[0,0,250]),     'b-', lw=2.5, label='Soft (0-250ppm)')
axes[3].plot(u_w, fuzz.trimf(u_w,[0,250,500]),   'g-', lw=2.5, label='Normal (0-500ppm)')
axes[3].plot(u_w, fuzz.trimf(u_w,[250,500,500]), 'r-', lw=2.5, label='Hard (250-500ppm)')
axes[3].axvline(x=water_input, color='black', lw=2, linestyle='--',
                label=f'Input={water_input}ppm ({water_label})')
axes[3].fill_between(u_w, fuzz.trimf(u_w,[0,0,250]),     alpha=0.08, color='blue')
axes[3].fill_between(u_w, fuzz.trimf(u_w,[0,250,500]),   alpha=0.08, color='green')
axes[3].fill_between(u_w, fuzz.trimf(u_w,[250,500,500]), alpha=0.08, color='red')
axes[3].set_title('Water Hardness Membership Function', fontsize=13, fontweight='bold')
axes[3].set_xlabel('Water Hardness (ppm)  [Range: 0–500 ppm]')
axes[3].set_ylabel('Membership Degree  μ(x)')
axes[3].set_xlim([-10, 510]); axes[3].set_ylim([-0.05, 1.15])
axes[3].legend(loc='upper right'); axes[3].grid(True, alpha=0.3)

#Graph 5:Washing Intensity Output
u_o = washing_intensity.universe
axes[4].plot(u_o, fuzz.trimf(u_o,[0,0,50]),    'b-', lw=2.5, label='Gentle (0-50%)')
axes[4].plot(u_o, fuzz.trimf(u_o,[0,50,100]),  'g-', lw=2.5, label='Normal (0-100%)')
axes[4].plot(u_o, fuzz.trimf(u_o,[50,100,100]),'r-', lw=2.5, label='Strong (50-100%)')
axes[4].axvline(x=crisp_output, color='black', lw=2.5, linestyle='--',
                label=f'Output={crisp_output:.2f}% → {out_label}')
axes[4].fill_between(u_o, fuzz.trimf(u_o,[0,0,50]),    alpha=0.08, color='blue')
axes[4].fill_between(u_o, fuzz.trimf(u_o,[0,50,100]),  alpha=0.08, color='green')
axes[4].fill_between(u_o, fuzz.trimf(u_o,[50,100,100]),alpha=0.08, color='red')
axes[4].set_title('Washing Intensity OUTPUT Membership Function',
                  fontsize=13, fontweight='bold')
axes[4].set_xlabel('Washing Intensity (%)  [0=No wash, 100=Maximum intensity]')
axes[4].set_ylabel('Membership Degree  μ(x)')
axes[4].set_xlim([-2, 102]); axes[4].set_ylim([-0.05, 1.15])
axes[4].legend(loc='upper right'); axes[4].grid(True, alpha=0.3)

plt.suptitle(#graph title
    f'Smart Washing Machine(Advanced)—Fuzzy Logic System\n'
    f'Input: Cloth={cloth_input}({cloth_label}),Load={load_input}kg({load_label}), '
    f'Dirtiness={dirt_input}({dirt_label}), Water={water_input}ppm({water_label})\n'
    f'Crisp Output = {crisp_output:.4f}%  →  Washing Intensity = {out_label}',
    fontsize=12, fontweight='bold', y=1.015
)

plt.tight_layout()
plt.savefig('smart_washing_machine_final.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGraph saved as 'smart_washing_machine_final.png'")